# Baseline real: TF-IDF + Logistic Regression

Este notebook implementa un baseline real para clasificación de comentarios tóxicos usando TF-IDF y regresión logística. El objetivo es superar el baseline trivial y establecer una referencia sólida para modelos más avanzados.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Importar módulos propios
from src.preprocessing import preprocess_for_tfidf
from src.augmentation import augment_text
from src.evaluation import (
        get_metrics, 
        compare_train_test, 
        display_comparison, 
        print_overfitting_analysis,
        print_report
)

In [2]:
# Cargar el dataset limpio (NO el augmentado, para evitar data leakage)
df = pd.read_csv('../data/processed/youtoxic_clean.csv')
print(f'Total de registros: {len(df)}')
print(f'Distribución de clases:')
print(df['IsToxic'].value_counts())
df.head()

Total de registros: 1000
Distribución de clases:
IsToxic
False    538
True     462
Name: count, dtype: int64


,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,if only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,dont you reckon them black lives matter banner...,True,True,False,False,True,False,False,False,False,False,False,False
3,there are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,the arab dude is absolutely right he should ha...,False,False,False,False,False,False,False,False,False,False,False,False


In [3]:
# Separar variables
X = df['Text']
y = df['IsToxic']

# Dividir en train y test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Tamaño train: {len(X_train)}, test: {len(X_test)}')

Tamaño train: 800, test: 200


## Preprocesamiento adicional para TF-IDF

Para modelos clásicos como TF-IDF + Logistic Regression, es recomendable:
- **Eliminar stopwords**: Palabras sin significado semántico (the, is, a...)
- **Lematización**: Reducir palabras a su raíz (running → run)
- **Convertir emojis a texto**: 😀 → "happy_face"

In [4]:
# Descargar recursos de NLTK (ejecutar PRIMERO, solo una vez)
import nltk
print("Descargando recursos de NLTK...")
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✅ Recursos descargados")

Descargando recursos de NLTK...
✅ Recursos descargados
✅ Recursos descargados


In [5]:
# La función preprocess_for_tfidf ahora viene de src/preprocessing.py
# Ejemplo de uso:
ejemplo = X_train.iloc[0]
print('Original:', ejemplo)
print('Procesado:', preprocess_for_tfidf(ejemplo))

Original: i wonder what the police expect will happen when they continually abuse and kill people do they honestly expect the public to do nothing about it to just lay back and take it and let the cops just turn into one massive street gang who are allowed to kill without consequence even a rat when its cornered will defend itself
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back take let cop turn one massive street gang allowed kill without consequence even rat cornered defend
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back take let cop turn one massive street gang allowed kill without consequence even rat cornered defend


In [6]:
# Aplicar preprocesamiento a train y test
print('Aplicando preprocesamiento a train...')
X_train_processed = X_train.apply(preprocess_for_tfidf)

print('Aplicando preprocesamiento a test...')
X_test_processed = X_test.apply(preprocess_for_tfidf)

print(f'\n✅ Preprocesamiento completado')
print(f'Train: {len(X_train_processed)} textos')
print(f'Test: {len(X_test_processed)} textos')

# Mostrar ejemplo
print('\n--- Ejemplo ---')
print('Original:', X_train.iloc[0])
print('Procesado:', X_train_processed.iloc[0])

Aplicando preprocesamiento a train...
Aplicando preprocesamiento a test...

✅ Preprocesamiento completado
Train: 800 textos
Test: 200 textos

--- Ejemplo ---
Original: i wonder what the police expect will happen when they continually abuse and kill people do they honestly expect the public to do nothing about it to just lay back and take it and let the cops just turn into one massive street gang who are allowed to kill without consequence even a rat when its cornered will defend itself
Procesado: wonder police expect happen continually abuse kill people honestly expect public nothing lay back take let cop turn one massive street gang allowed kill without consequence even rat cornered defend

✅ Preprocesamiento completado
Train: 800 textos
Test: 200 textos

--- Ejemplo ---
Original: i wonder what the police expect will happen when they continually abuse and kill people do they honestly expect the public to do nothing about it to just lay back and take it and let the cops just turn into 

## ⚠️ Orden Correcto del Pipeline

**Es crucial seguir este orden para evitar Data Leakage:**

1. ✅ Cargar datos limpios
2. ✅ **SPLIT Train/Test** (antes de augmentation)
3. ✅ Data Augmentation **SOLO en Train**
4. ✅ Vectorización (fit en Train, transform en ambos)
5. ✅ Entrenamiento y evaluación

> Si aumentamos los datos ANTES de dividir, podríamos tener el texto original en Test y su variación en Train. El modelo "haría trampa" memorizando respuestas.

## Data Augmentation (Solo en Train)

Aplicamos técnicas de augmentation SOLO al conjunto de entrenamiento:
- **Back-translation**: Traduce a otro idioma y vuelve (paráfrasis naturales)
- **Random Swap**: Intercambia posiciones de palabras

In [7]:
# Las funciones de augmentation ahora vienen de src/augmentation.py
# Disponibles: back_translation, random_swap, synonym_replacement, augment_text

print("Funciones de augmentation importadas de src/augmentation.py ✓")
print("  - back_translation (español)")
print("  - random_swap")
print("  - synonym_replacement (WordNet)")
print("  - augment_text (aplica una técnica aleatoria)")

Funciones de augmentation importadas de src/augmentation.py ✓
  - back_translation (español)
  - random_swap
  - synonym_replacement (WordNet)
  - augment_text (aplica una técnica aleatoria)


In [ ]:
# Aplicar augmentation 2X a TODOS los datos del TRAIN (ya preprocesados)
# Más augmentation = más variabilidad = mejor generalización

import random
random.seed(42)  # Fijar semilla para reproducibilidad

train_df = pd.DataFrame({'Text': X_train_processed, 'IsToxic': y_train}) # type: ignore

print(f'Train original: {len(train_df)}')
print(f'Distribución original: {train_df["IsToxic"].value_counts().to_dict()}')

# Generar 2 versiones aumentadas por cada texto
augmented_texts = []
augmented_labels = []

print('\nAplicando augmentation 2X a todo el train...')
for i, (text, label) in enumerate(zip(train_df['Text'], train_df['IsToxic'])):
    # Generar 2 versiones aumentadas por texto
    augmented_texts.append(augment_text(text))  # type: ignore
    augmented_labels.append(label)
    augmented_texts.append(augment_text(text))  # type: ignore
    augmented_labels.append(label)
    if (i + 1) % 200 == 0:
        print(f'  Procesados {i + 1}/{len(train_df)}...')

# Combinar train original + augmentados (ahora 3x el tamaño original)
df_augmented = pd.DataFrame({'Text': augmented_texts, 'IsToxic': augmented_labels})  # type: ignore
train_final = pd.concat([train_df, df_augmented], ignore_index=True) # type: ignore
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

# Actualizar X_train e y_train
X_train_aug = train_final['Text']
y_train_aug = train_final['IsToxic']

print(f'\nTrain después de augmentation 2X: {len(train_final)} (3x original)')
print(f'Distribución final: {train_final["IsToxic"].value_counts().to_dict()}')

Train original: 800
Distribución original: {False: 430, True: 370}

Aplicando augmentation 2X a todo el train...
  Procesados 200/800...
  Procesados 200/800...
  Procesados 400/800...
  Procesados 400/800...
  Procesados 600/800...
  Procesados 600/800...
  Procesados 800/800...

Train después de augmentation 2X: 2400 (3x original)
Distribución final: {False: 1290, True: 1110}
  Procesados 800/800...

Train después de augmentation 2X: 2400 (3x original)
Distribución final: {False: 1290, True: 1110}


In [9]:
# Crear y ajustar el vectorizador TF-IDF (fit SOLO en train augmentado)
# Configuración AÚN MÁS SIMPLE para reducir overfitting
vectorizer = TfidfVectorizer(
    max_features=500,   # Solo 500 palabras (menos features = menos overfitting)
    min_df=3,           # Ignorar palabras que aparecen en menos de 3 documentos
    max_df=0.90,        # Ignorar palabras que aparecen en más del 90% de documentos
    ngram_range=(1, 1)  # SOLO unigramas (sin bigramas)
)
X_train_tfidf = vectorizer.fit_transform(X_train_aug)  # Fit en train augmentado
X_test_tfidf = vectorizer.transform(X_test_processed)  # Transform en test preprocesado
print(f'Shape train: {X_train_tfidf.shape}, test: {X_test_tfidf.shape}')
print(f'Vocabulario reducido a {len(vectorizer.vocabulary_)} palabras')

Shape train: (2400, 500), test: (200, 500)
Vocabulario reducido a 500 palabras


In [10]:
# Entrenar con regularización óptima (mejor balance overfitting/performance)
clf = LogisticRegression(
    max_iter=1000, 
    random_state=42,
    C=0.005,             # Regularización óptima (gap ~9.9%)
    class_weight='balanced'
)

clf.fit(X_train_tfidf, y_train_aug)
print(f'Modelo entrenado con {len(y_train_aug)} muestras')
print(f'Regularización C={clf.C} (óptima), class_weight={clf.class_weight}')

Modelo entrenado con 2400 muestras
Regularización C=0.005 (óptima), class_weight=balanced


In [ ]:
# Predecir en train augmentado
y_train_pred = clf.predict(X_train_tfidf)  # type: ignore

# Calcular métricas en train usando src/evaluation.py
metrics_train = get_metrics(y_train_aug, y_train_pred)  # type: ignore
print('--- Métricas en entrenamiento (augmentado) ---')
for metric, value in metrics_train.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_train_aug, y_train_pred, "Clasificación Train")  # type: ignore

--- Métricas en entrenamiento (augmentado) ---
Accuracy: 0.8017
Precision: 0.8084
Recall: 0.7486
F1: 0.7774

--- Clasificación Train ---
              precision    recall  f1-score   support

       False       0.80      0.85      0.82      1290
        True       0.81      0.75      0.78      1110

    accuracy                           0.80      2400
   macro avg       0.80      0.80      0.80      2400
weighted avg       0.80      0.80      0.80      2400



In [ ]:
# Predecir en test
y_pred = clf.predict(X_test_tfidf)  # type: ignore

# Calcular métricas usando src/evaluation.py
metrics_test = get_metrics(y_test, y_pred)  # type: ignore
print('--- Métricas en test ---')
for metric, value in metrics_test.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_test, y_pred, "Clasificación Test")  # type: ignore

--- Métricas en test ---
Accuracy: 0.7250
Precision: 0.7229
Recall: 0.6522
F1: 0.6857

--- Clasificación Test ---
              precision    recall  f1-score   support

       False       0.73      0.79      0.76       108
        True       0.72      0.65      0.69        92

    accuracy                           0.72       200
   macro avg       0.72      0.72      0.72       200
weighted avg       0.72      0.72      0.72       200



In [ ]:
# Cargar predicciones del baseline trivial y calcular métricas
df_baseline = pd.read_csv('../data/processed/baseline_trivial_pred.csv') # type: ignore
y_true_baseline = df_baseline['IsToxic']
y_pred_baseline = df_baseline['baseline_pred']

# Métricas usando src/evaluation.py
metrics_trivial = get_metrics(y_true_baseline, y_pred_baseline) # type: ignore
print('--- Baseline trivial ---')
for metric, value in metrics_trivial.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_true_baseline, y_pred_baseline, "Clasificación Baseline Trivial")    # type: ignore

--- Baseline trivial ---
Accuracy: 0.5380
Precision: 0.0000
Recall: 0.0000
F1: 0.0000

--- Clasificación Baseline Trivial ---
              precision    recall  f1-score   support

       False       0.54      1.00      0.70       538
        True       0.00      0.00      0.00       462

    accuracy                           0.54      1000
   macro avg       0.27      0.50      0.35      1000
weighted avg       0.29      0.54      0.38      1000



In [ ]:
# Comparar métricas de baseline trivial y modelo TF-IDF + Logistic Regression
# La función get_metrics ahora viene de src/evaluation.py

# Métricas baseline trivial
metrics_baseline = get_metrics(y_true_baseline, y_pred_baseline)  # type: ignore
# Métricas modelo real
metrics_model = get_metrics(y_test, y_pred)  # type: ignore

report = pd.DataFrame([metrics_baseline, metrics_model],     #type: ignore  
                    index=['Baseline trivial', 'TF-IDF + LogReg']) 
print('Comparativa de métricas:')
display(report) # type: ignore

Comparativa de métricas:


,accuracy,precision,recall,f1
Baseline trivial,0.538,0.000000,0.000000,0.000000
TF-IDF + LogReg,0.725,0.722892,0.652174,0.685714


In [ ]:
# Comparar métricas de entrenamiento y test usando src/evaluation.py
comparison = compare_train_test(y_train_aug, y_train_pred, y_test, y_pred)  # type: ignore

print('📊 Comparativa de métricas (Train vs Test):')
display(display_comparison(comparison))

# Análisis de overfitting
print_overfitting_analysis(comparison)

📊 Comparativa de métricas (Train vs Test):


,accuracy,precision,recall,f1
Train,0.801667,0.808366,0.748649,0.777362
Test,0.725000,0.722892,0.652174,0.685714
Gap (Train-Test),0.076667,0.085474,0.096475,0.091648



🎯 Análisis de Overfitting:
⚠️ Gap F1 = 0.092 (>5%) - Posible overfitting


## Análisis de la comparación entre métricas de entrenamiento y test

A continuación se muestra un breve análisis sobre las diferencias entre las métricas de entrenamiento y test. Si las métricas de entrenamiento son mucho mayores que las de test, puede indicar overfitting. Si ambas son bajas, puede indicar underfitting o un modelo poco expresivo.


## Conclusiones de la comparación

- El baseline trivial predice siempre la clase mayoritaria y sirve como referencia mínima.
- El modelo TF-IDF + Logistic Regression utiliza el texto y, si las métricas son mejores que el baseline, demuestra que el modelo aprende patrones útiles.
- Si tu modelo supera al baseline en F1-score, accuracy, precision y recall, puedes avanzar a modelos más complejos o probar mejoras.
- Si no lo supera, revisa el preprocesamiento, los datos o prueba otros enfoques.

> **Recuerda:** El objetivo es que cualquier modelo real supere claramente al baseline trivial para justificar su uso.